In [1]:
# Load env variables and create client
from anthropic import Anthropic

In [2]:
client = Anthropic(
    base_url="http://localhost:11211/api/anthropic",
    auth_token="not-used"
)

In [3]:
# Get all models from GenAI bridge with:
# curl -s http://localhost:11211/api/openai/v1/models | jq -r '.data[].id' | grep -i claude
model = "aws:anthropic.claude-haiku-4-5-20251001-v1:0"

In [4]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [47]:
import json


def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing a task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
            "format": "Expected format. One of 'python', 'json' or 'regex'",
            "solution_criteria": "Description of the characteristics of a good solution"
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    # Prefill the assistant turn with "```json" so Claude continues from an open
    # code fence and emits JSON immediately, with no conversational preamble.
    add_assistant_message(messages, "```json")
    # Set stop_sequences=["```"] so generation halts at the closing fence,
    # leaving `text` as clean JSON ready for json.loads — no parsing required.
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

Modern alternatives to prefill + stop:

1. Tool use — Define a tool with a JSON schema and force Claude to call it (tool_choice). You get validated, structured output directly, with the SDK handling parsing.
2. Provider structured-output features — Some SDKs expose a response_format / JSON-mode parameter that constrains the model to valid JSON without needing a tool definition.

Prefill + stop remains useful as a lightweight fallback, especially for non-JSON formats (XML, Markdown, code snippets) where defining a schema is overkill.m

In [48]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}

    * Respond only in Python, JSON, or plain Regex
    * Do not add any comments or commentary explanation.
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, f"```{test_case['format']}")
    output = chat(messages, stop_sequences=["```"])
    return output

In [49]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    <task>
    {test_case["task"]}
    </task>

    <solution>
    {output}
    </solution>

    <solution_criteria>
    {test_case["solution_criteria"]}
    </solution_criteria>
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct. 
    Consider the solution criteria provided in your judgements.
    Example response shape:
    {{
        "strengts": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number,
    }}
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [50]:
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(test_case, response):
    format = test_case["format"]
    match format:
        case "json":
            return validate_json(response)
        case "python":
            return validate_python(response)
        case "regex":
            return validate_regex(response)
        

In [51]:
def run_test_case(test_case):
    """Calls run_prompt over a test case, then grades the result"""
    output = run_prompt(test_case)

    syntax_score = grade_syntax(test_case, output)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    score = (syntax_score + model_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [52]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [53]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [54]:
evals = run_eval(dataset)

with open('responses.json', 'w') as f:
    json.dump(evals, f, indent=2)

Average score: 8.5
